# Qa fetch routing

Purpose: inspect the named geometry, dataset, or model diagnostic.

Prerequisites: install the project with the notebook extra and supply the research artifacts selected in the configuration cells. Launch Jupyter from the project root. See `docs/notebooks.md` for per-notebook inputs.

Outputs: displayed diagnostics and, where configured, exported figures/tables. Run cells from top to bottom. Saved outputs have been cleared.


# QA: Boundary Curtain Fetch Routing

This notebook validates the routing features generated by `src/fetch_router.py` by overlaying:
- the EPSG:32633 bathymetry subgrid,
- the offshore boundary curtain,
- shortest over-water nearshore paths routed inward from the curtain.


In [ ]:
from __future__ import annotations

import pickle
from pathlib import Path

import contextily as ctx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import Markdown, display

plt.rcParams["figure.dpi"] = 240
TARGET_EPSG = 32633


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for parent in [cwd, *cwd.parents]:
        if all((parent / name).exists() for name in ("configs", "data", "notebooks", "src")):
            return parent
    raise FileNotFoundError("Could not resolve project root with configs/data/notebooks/src.")


def load_bathy(nc_path: Path) -> tuple[np.ndarray, np.ndarray, np.ndarray, list[float]]:
    with xr.open_dataset(nc_path) as ds:
        if "z" in ds.data_vars:
            da = ds["z"]
        elif "depth" in ds.data_vars:
            da = ds["depth"]
        else:
            raise KeyError("Expected bathymetry variable `z` or `depth`.")

        x = np.asarray(da["x"].to_numpy(), dtype=float)
        y = np.asarray(da["y"].to_numpy(), dtype=float)
        z = np.asarray(da.transpose("y", "x").to_numpy(), dtype=float)

    extent = [float(np.nanmin(x)), float(np.nanmax(x)), float(np.nanmin(y)), float(np.nanmax(y))]
    return x, y, z, extent


PROJECT_ROOT = resolve_project_root()
SUBGRID_NC = PROJECT_ROOT / "data/bathy/subgrid_padded.nc"
ROUTING_PKL = PROJECT_ROOT / "data/processed/routing_features.pkl"

x_vals, y_vals, z_grid, extent = load_bathy(SUBGRID_NC)
with ROUTING_PKL.open("rb") as handle:
    routing = pickle.load(handle)

routes_df = routing.get("routes", pd.DataFrame())
if not isinstance(routes_df, pd.DataFrame):
    routes_df = pd.DataFrame(routes_df)

curtain_xy = np.asarray(routing.get("curtain_xy", []), dtype=float)

print(f"Project root: {PROJECT_ROOT}")
print(f"Target CRS: EPSG:{TARGET_EPSG}")
print(f"Grid shape (y, x): {z_grid.shape}")
print(f"Route rows: {len(routes_df)}")
print(f"Extent: {extent}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
ax.set_xlim(extent[0], extent[1])
ax.set_ylim(extent[2], extent[3])

try:
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, crs="EPSG:32633", zoom=12)
except Exception as exc:
    print(f"Basemap fetch failed ({exc}); plotting without imagery.")

z_masked = np.ma.masked_invalid(z_grid)
img = ax.imshow(
    z_masked,
    extent=extent,
    origin="lower",
    cmap="Blues_r",
    alpha=0.0,
    interpolation="nearest",
    zorder=2,
)

if curtain_xy.size:
    ax.plot(
        curtain_xy[:, 0],
        curtain_xy[:, 1],
        color="#ffe600",
        linewidth=2.4,
        label="Boundary curtain",
        zorder=4,
    )

for _, row in routes_df.iterrows():
    path_xy = np.asarray(row.get("path_xy", []), dtype=float)
    if path_xy.ndim == 2 and path_xy.shape[0] >= 2:
        ax.plot(
            path_xy[:, 0],
            path_xy[:, 1],
            color="#00d4ff",
            linewidth=1.5,
            alpha=0.88,
            zorder=3,
        )


ax.set_title("Boundary Curtain and Reverse-Routed Nearshore Paths (EPSG:32633)")
ax.set_xlabel("X (m), EPSG:32633")
ax.set_ylabel("Y (m), EPSG:32633")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
length_table = (
    routes_df[["site_name", "path_length_m", "path_point_count", "snap_distance_m", "reachable"]]
    .sort_values("site_name")
    .reset_index(drop=True)
)

display(Markdown("### Nearshore Path Lengths"))
display(Markdown(length_table.to_markdown(index=False, floatfmt=".2f")))
length_table